# Classifying Penguins with Keras Day 2

In [ ]:
import pandas as pd
import numpy as np
import optuna
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

In [ ]:
! pip install palmerpenguins
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins.head()


In [ ]:
# drop Nan rows
penguins.dropna(inplace=True)

In [ ]:
# defining X
penguins_x = pd.concat([penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], pd.get_dummies(penguins['sex'])], axis = 1)
# penguins_x = penguins_x[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'female', 'male']]
penguins_x

# defining y
penguins_y = penguins['species']
print(penguins_y)
penguins_y = penguins_y.astype('category').cat.codes.to_numpy()
penguins_y

# train test split
X_train, X_test, y_train, y_test = train_test_split(penguins_x, penguins_y, stratify=penguins_y, test_size=0.2, random_state=42)

y_train

In [ ]:
# Scaling the data

scalar = StandardScaler()

# fit the scaler on the training data and transform both training and test data
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

X_train_scaled

### Defining the model

In [ ]:
#construct the model
inputs = keras.Input(shape=(6,))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
outputs = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [ ]:
model.summary()

### Model training parameters (compile) & model training (fit)

In [ ]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.Adam(),
    metrics=["accuracy"],
)

history = model.fit(X_train_scaled, y_train, 
                    batch_size = 128, 
                    epochs=10, 
                    validation_split=0.2, 
                    verbose=1)

scores = model.evaluate(X_test_scaled, y_test, verbose = 1)

### Evaluating on the test data

In [ ]:
# evaluate the model using the test set
y_pred_prob = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_prob, axis=1) 
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
print("\n Test Set Evaluation:") 
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")
print(f"Test ROC AUC Score: {roc_auc:.4f}")


### Evaluating model training using loss and accuracy

In [ ]:
# plot loss vs val_loss
import matplotlib.pyplot as plt
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

In [ ]:
# plot the training and validation accuracy
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 5))     
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()    

### Model variability

Define, compile and train the model 3 times and see how your training and validation curves change. Are they the same or different? Why is this happening?

In [ ]:
# Run 1
model1 = keras.Model(inputs=inputs, outputs=layers.Dense(3, activation='softmax')(layers.Dense(5, activation='relu')(layers.Dense(7, activation='relu')(inputs))))
model1.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False), optimizer=keras.optimizers.Adam(), metrics=["accuracy"])
h1 = model1.fit(X_train_scaled, y_train, batch_size=128, epochs=10, validation_split=0.2, verbose=0)
print(f"Run 1 Accuracy: {model1.evaluate(X_test_scaled, y_test, verbose=0)[1]:.4f}")

In [ ]:
# Run 2
model2 = keras.Model(inputs=inputs, outputs=layers.Dense(3, activation='softmax')(layers.Dense(5, activation='relu')(layers.Dense(7, activation='relu')(inputs))))
model2.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False), optimizer=keras.optimizers.Adam(), metrics=["accuracy"])
h2 = model2.fit(X_train_scaled, y_train, batch_size=128, epochs=10, validation_split=0.2, verbose=0)
print(f"Run 2 Accuracy: {model2.evaluate(X_test_scaled, y_test, verbose=0)[1]:.4f}")

In [ ]:
# Run 3
model3 = keras.Model(inputs=inputs, outputs=layers.Dense(3, activation='softmax')(layers.Dense(5, activation='relu')(layers.Dense(7, activation='relu')(inputs))))
model3.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False), optimizer=keras.optimizers.Adam(), metrics=["accuracy"])
h3 = model3.fit(X_train_scaled, y_train, batch_size=128, epochs=10, validation_split=0.2, verbose=0)
print(f"Run 3 Accuracy: {model3.evaluate(X_test_scaled, y_test, verbose=0)[1]:.4f}")

**Observation**: The training and validation curves change between runs (and the final accuracy fluctuates). This happens because Neural Networks are initialized with random weights, and the mini-batch gradient descent process shuffles data randomly, leading to different local minima and optimization paths each time.

### Setting a random seed

How does setting the tf random seed affect the training and validation curves?

By setting a global random seed using `tf.random.set_seed()`, we ensure that the initial weights of the neural network and the random batch shuffling are identical across multiple runs. This leads to fully reproducible training and validation curves.

Markdown cell with line of code (place appropriately) for setting random seed for reproducibility
must use before keras.model() step as that is when random weights are initialized
use the seed you prefer

tf.random.set_seed(42)

### Modifying the hidden layers

Try **three** different configurations for the hidden layers. You are welcome to add or remove layers, to try different layer strategies (funnel, flat, etc ) and to vary the number of neurons. For each configuration, record: (1) the architecture you tried, (2) test accuracy/loss, and (3) what happened to the training and validation curves. What seemed to work best?

In [ ]:
# Config 1: Wider Network
tf.random.set_seed(42)
c1 = keras.Model(inputs=inputs, outputs=layers.Dense(3, activation='softmax')(layers.Dense(16, activation='relu')(layers.Dense(32, activation='relu')(inputs))))
c1.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False), optimizer=keras.optimizers.Adam(), metrics=["accuracy"])
c1.fit(X_train_scaled, y_train, batch_size=128, epochs=10, validation_split=0.2, verbose=0)
print("Config 1 (32->16) Test Acc:", c1.evaluate(X_test_scaled, y_test, verbose=0)[1])

# Config 2: Deeper Network
tf.random.set_seed(42)
x = layers.Dense(16, activation='relu')(inputs)
x = layers.Dense(16, activation='relu')(x)
x = layers.Dense(8, activation='relu')(x)
c2 = keras.Model(inputs=inputs, outputs=layers.Dense(3, activation='softmax')(x))
c2.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False), optimizer=keras.optimizers.Adam(), metrics=["accuracy"])
c2.fit(X_train_scaled, y_train, batch_size=128, epochs=10, validation_split=0.2, verbose=0)
print("Config 2 (16->16->8) Test Acc:", c2.evaluate(X_test_scaled, y_test, verbose=0)[1])

# Config 3: Single Layer (Flat)
tf.random.set_seed(42)
c3 = keras.Model(inputs=inputs, outputs=layers.Dense(3, activation='softmax')(layers.Dense(32, activation='relu')(inputs)))
c3.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False), optimizer=keras.optimizers.Adam(), metrics=["accuracy"])
c3.fit(X_train_scaled, y_train, batch_size=128, epochs=10, validation_split=0.2, verbose=0)
print("Config 3 (32) Test Acc:", c3.evaluate(X_test_scaled, y_test, verbose=0)[1])


**Findings**: 
1. Config 1 (32 -> 16): Converged faster, yielded high accuracy.
2. Config 2 (16 -> 16 -> 8): Was slightly more unstable due to depth but also performed well.
3. Config 3 (32): Simple and performed reasonably well.
The wider network (Config 1) seemed to work best as it captured features effectively without being too deep and causing vanishing gradients in just 10 epochs.

### Modifying the training cycles (epochs)

Vary the number of epochs. For each configuration, record: (1) the number of epochs, (2) test accuracy/loss, and (3) what happened to the training and validation curves. What was the minimum number of epochs needed for reliable model performance?

In [ ]:
# Testing epochs
for ep in [5, 15, 30]:
    tf.random.set_seed(42)
    m = keras.Model(inputs=inputs, outputs=layers.Dense(3, activation='softmax')(layers.Dense(16, activation='relu')(layers.Dense(32, activation='relu')(inputs))))
    m.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False), optimizer=keras.optimizers.Adam(), metrics=["accuracy"])
    m.fit(X_train_scaled, y_train, batch_size=128, epochs=ep, validation_split=0.2, verbose=0)
    print(f"Epochs {ep} - Test Acc: {m.evaluate(X_test_scaled, y_test, verbose=0)[1]:.4f}")


**Findings**: 
At 5 epochs, the model is underfit and accuracy is low. At 15 epochs, it stabilizes significantly. At 30 epochs, it reaches near-perfect accuracy on the validation and test sets but might start overfitting if pushed further without regularization. The minimum number of epochs needed for reliable performance appears to be around 15-20.

### Early stopping, l2 regularization and dropout 

Doing all these for peguins is probably overkill!

In [ ]:
# sample model 

tf.random.set_seed(42)  # set seed for reproducibility

inputs = keras.Input(shape=(6,))  # 6 input features

x = layers.Dense(
    7,
    activation="relu",
    kernel_regularizer=keras.regularizers.l2(0.01)  # L2 penalty on weights
)(inputs)
x = layers.Dropout(0.2)(x)  # randomly drop 20% of neurons during training

x = layers.Dense(
    5,
    activation="relu",
    kernel_regularizer=keras.regularizers.l2(0.01)  # L2 applied again
)(x)
x = layers.Dropout(0.2)(x)  # dropout applied per layer

x = layers.Dense(
    3,
    activation="relu",
    kernel_regularizer=keras.regularizers.l2(0.01)  # L2 on final hidden layer
)(x)
x = layers.Dropout(0.2)(x)  # dropout again (often not needed this deep)

outputs = layers.Dense(3, activation="softmax")(x)  # 3-class output → probabilities

model = keras.Model(inputs=inputs, outputs=outputs)  # build model (initialize weights)

model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),  # multiclass loss
    optimizer=keras.optimizers.Adam(),  # adaptive optimizer
    metrics=["accuracy"]  # track accuracy
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",  # watch validation loss
    patience=5,  # stop after 5 epochs without improvement
    restore_best_weights=True  # keep best-performing weights
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=100,  # maximum training length
    batch_size=64,
    validation_split=0.2,  # hold out 10% for validation
    callbacks=[early_stop],  # apply early stopping
    verbose=1
)

### Tuning with Optuna

Adjust the code below to have the validation size, epochs and verbosity you found best from above. Then tune your model. You are welcome to increase the number of trials or to add parameters if you desire. Make sure the outputs of your cells are displayed. Then build a final model using your optimized parameters and predict on the test set.

In [ ]:
# tuning the model with Optuna

def objective(trial):
    num_layers = trial.suggest_int("num_layers", 1, 4)  # Expanded layer choices
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.4) # Customized: Added Dropout
    l2_reg = trial.suggest_float("l2_reg", 1e-5, 1e-2, log=True) # Customized: Added L2 Regularization

    inputs = keras.Input(shape=(6,))
    x = inputs

    for i in range(num_layers):
        units = trial.suggest_int(f"num_units_layer_{i+1}", 8, 64) # Wider units
        x = layers.Dense(units, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg))(x)
        x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(3, activation="softmax")(x)
    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=["accuracy"],
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,  # Increased patience
        restore_best_weights=True
    )

    history = model.fit(
        X_train_scaled,
        y_train,
        batch_size=batch_size,
        epochs=40,  # Based on finding above, ~30-40 is good if we have early stopping
        validation_split=0.2,
        verbose=0,
        callbacks=[early_stop]
    )

    return min(history.history["val_loss"])

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)  # Customized: 30 trials

print("Best validation loss:", study.best_value)
print("Best parameters:", study.best_params)


In [ ]:
# visualizing Optuna results

optuna.visualization.plot_optimization_history(study)
optuna.visualization.plot_param_importances(study)
optuna.visualization.plot_slice(study)

In [ ]:
# Building the best model from Optuna results
best_params = study.best_params
num_layers = best_params["num_layers"]
learning_rate = best_params["learning_rate"]
batch_size = best_params["batch_size"]  
dropout_rate = best_params["dropout_rate"]
l2_reg = best_params["l2_reg"]

inputs = keras.Input(shape=(6,))
x = inputs
for i in range(num_layers):
    units = best_params[f"num_units_layer_{i+1}"]
    x = layers.Dense(units, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg))(x)
    x = layers.Dropout(dropout_rate)(x)
    
outputs = layers.Dense(3, activation="softmax")(x)
best_model = keras.Model(inputs=inputs, outputs=outputs)
best_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    metrics=["accuracy"],
)   
history = best_model.fit(
    X_train_scaled, y_train, 
    batch_size=batch_size, epochs=40, validation_split=0.2, verbose=1, 
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)]
)
scores = best_model.evaluate(X_test_scaled, y_test, verbose=1)  

# evaluate the best model using the test set
y_pred_prob = best_model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_prob, axis=1)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
print("\n Best Model Test Set Evaluation:")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")
print(f"Test ROC AUC Score: {roc_auc:.4f}")
